# This notebook extracts realizations to be exported into JutulDarcy

* For JutulDarcy, set reverse_j = False when using generate_full_properties(). 
* The finit files must contain ACTID to identify active cells, in addition to properties such as PORO and PERMX.
* This notebook has been validated by (1) processing a realization using this notebook, (2) import processed realiztion into Petrel, (3) compare it with the realization in Petrel and found statistics are exactly the same. Note when exporting a property out of Petrel, setting undefined cells to be 0 (instead of -999) would generate the exact same file as a processed realization by this notebook.

## Step 1: Extract finit files from simulation cases

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from extract_filenames_files import extract_sorted_files

################# Start of user inputs ########## 
data_folder_path = repo_root/'data'/'sim_cases' # path to the folder containing simulation case subfolders, each with a .FINIT file
results_folder_path = repo_root/'results'/'finit_files' # path to the folder where extracted .FINIT files will be copied. Will be created if it doesn't exist.
################## End of user inputs  ##########

extract_sorted_files(
        folder_dir = data_folder_path,
        save_dir = results_folder_path,
        file_extension = '.FINIT',
        show_summary = True,
        show_filenames = False
)

Extracting files: 4it [00:01,  3.77it/s]

Total .FINIT files copied: 3


## Step 2: Extract properties

In [ ]:
import sys
from pathlib import Path
repo_root = Path.cwd().parent
sys.path.append(str(repo_root / "src"))

from extract_properties_from_finit import extract_properties_from_finit
from generate_full_properties import generate_full_properties
from CMG_format_compress import CMG_format_compress
from extract_filenames_files import extract_sorted_filenames

################# Start of user inputs ########## 
finit_folder_path = repo_root/'results'/'finit_files' # path to the folder containing .FINIT files extracted from simulation cases
save_folder_path = repo_root/'results'/'properties' # path to the folder where extracted properties will be saved. Will be created if it doesn't exist.
grid_shape = (139, 248, 23) # shape of the geomodel grid (i, j, k)
property_list = ['PORO', 'PERMX'] # list of property keywords to extract from
################## End of user inputs  ##########

save_folder_path.mkdir(exist_ok=True)

# extract finit file names from its folder
finit_file_names = extract_sorted_filenames(
        folder_dir = finit_folder_path,
        is_save = False,
        save_dir = None,
        save_name = None
)

# extract properties
for finit_file_name in finit_file_names:
    finit_file_path = finit_folder_path / finit_file_name
    # some finit files contains PORO cells < active cells, so use try-except-continue to avoid interuption
    try:
        # STEP 1: Extract properties from FINIT file for active cells only
        extracted_property_dict = extract_properties_from_finit(
            finit_file_path = finit_file_path,
            keywords = property_list + ['ACTID'],
            is_save = False,
            save_dir = save_folder_path,
            save_name = finit_file_name.split('.')[0],
            show_summary = False
        )

        # STEP 2: Generate full properties for all cells (fill inactive cells with zeros)
        full_property_dict = generate_full_properties(
            property_dict = extracted_property_dict,
            property_list = property_list, 
            grid_shape = grid_shape,
            is_save = False,
            save_dir = save_folder_path,
            save_name = finit_file_name.split('.')[0],
            show_summary = False,
            reverse_j = False
            )


        # STEP 3: Compress full properties to CMG format (repeated values as N*value)
        for key in property_list:
            CMG_format_compress(
                array = full_property_dict[key], 
                keyword = key, 
                max_line_length = 80,
                show_summary = False,
                save_dir = save_folder_path,
                save_name = finit_file_name.split('.')[0]
            )
    except ValueError as e:
        print(f"Warning: Skipping '{finit_file_name}'. Reason: {e}")
        continue
